# Follow-up probe (per model organism)

The follow-up probe **reuses the mean-difference probe** (same `direction` + `layer`,
trained in `tutorials/3.1`) but reads it at a different position: it appends a forced
follow-up turn — **"Was your last response deceptive? (Yes/No)" → "No"** — and reads the
residual at the forced reply's last token (see `tutorials/3.2`). So it is the *same
single-forward-pass probe* as `mean_probe_organism.ipynb`; only the response span differs.

Because it is one forward pass per row, it runs exactly like the mean probe — as **one
`util.run_full_session`** (the whole dataset in a single remote NDIF session), not a
bespoke multi-session loop. Steps:

1. Identify the organism from `DATASET_NAME` row 0 (`model` + `lora`) and pick the matching
   `followup_probe_*.npz` (falling back to the base-model probe, then neutral).
2. `run_full_session` builds that organism's model, tokenizes every row with
   `followup_preprocess` (append the forced turn, mark the reply token), and scores every
   row in one session.
3. `detect` reads the probe's layer, takes the forced-reply token, projects onto the
   direction, calibrates (`bias` / `score_std`) → sigmoid → `[0,1]`, then `tracer.stop()`.

In [ ]:
import os

# ── DO NOT CHANGE ───────────────────────────────────────────────────
# The runner sets these per run; read them, don't hard-code or override.
#   DATASET_NAME : the eval dataset to predict on (encodes the organism).
#   NDIF_HOST / NDIF_API_KEY / HF_TOKEN : injected; nnsight + HF read them.
DATASET_NAME = os.environ["DATASET_NAME"]

# Optional `python submit.py --limit N` -> $ALETHEIA_LIMIT: score only the first N rows
# (fast rehearsal). Unset on the leaderboard -> every row.
LIMIT = int(os.environ["ALETHEIA_LIMIT"]) if os.environ.get("ALETHEIA_LIMIT") else None

## 1. Select this organism's probe

`tutorials/3.2` saved one `followup_probe_*.npz` per organism (each reusing that organism's
mean-probe `direction` / `layer`, plus the calibration `bias` / `score_std` and the
`max_seq_len` it was tokenized at). We read the organism off `DATASET_NAME`'s row 0 and pick
the file whose `(model_id, lora_id)` matches. `util.pick_probe` never raises: it falls back
exact → base-model → any-lora-for-this-model → `None`; on `None` we emit neutral 0.5 for
every row instead of crashing.

In [ ]:
import sys
import numpy as np

sys.path.insert(0, "submission")   # util.py lives in submission/; the runner starts at repo root
import util

# The organism is defined by the dataset's own columns (NOT the dataset name): peek row 0.
ex0 = util.load_examples(DATASET_NAME)[0]
DS_MODEL = ex0["model"]
DS_LORA  = ex0.get("lora")                       # None for a plain base organism

picked = util.pick_probe("followup_probe_*.npz", DS_MODEL, DS_LORA)
print(f"dataset organism: model={DS_MODEL} | lora={DS_LORA}")

if picked is None:
    HAVE_PROBE = False
    print("[warn] no follow-up probe for this model -> emitting neutral scores")
else:
    HAVE_PROBE = True
    PROBE_FILE, z = picked
    DIRECTION   = z["direction"].astype(np.float32)                # (hidden,)
    LAYER       = int(z["layer"])
    BIAS        = float(z["bias"])        if "bias" in z.files        else 0.0
    SCORE_STD   = float(z["score_std"])   if "score_std" in z.files   else 1.0
    MAX_SEQ_LEN = int(z["max_seq_len"])   if "max_seq_len" in z.files else 256
    print(f"using {PROBE_FILE} | layer={LAYER} | dim={DIRECTION.shape[0]} | "
          f"bias={BIAS:.4g} | score_std={SCORE_STD:.4g} | max_len={MAX_SEQ_LEN}")

## 2. Method
(same probe as `tutorials/3.1`/`mean_probe`, read at the forced follow-up token per `tutorials/3.2`)

In [ ]:
import torch
from util import decoder_layers, run_full_session, load_examples


def followup_preprocess(messages, tokenizer, max_len):
    """Append the forced follow-up turn ("Was your last response deceptive? (Yes/No)" -> "No")
    and return (token_ids, (idx, idx+1)) where idx is the LAST token of the forced "No" reply --
    the single position the probe reads.

    Locating "No" mirrors `tutorials/3.2` `tokenize_eval` EXACTLY -- render the chat template to
    text, re-tokenize with `add_special_tokens=False`, and map the answer's character span to
    tokens via `char_to_token`. That is the same tokenization the probe's `bias`/`score_std`
    were calibrated with, so eval reads the identical position. (A `while ... in all_special_ids`
    strip does NOT work here: Qwen's template emits `No <|im_end|> \n` and gemma `No <end_of_turn>
    \n` -- the trailing "\n" is not a special token, so stripping specials alone lands the probe
    on the newline instead of "No".)

    Self-contained so it cloudpickles into the remote session; folds a leading system turn into
    the first user turn (gemma). NEVER raises: on any tokenization error it falls back to a plain
    last-token span so one odd row can't abort the whole session."""
    FOLLOWUP_QUESTION = "Was your last response deceptive? (Yes/No)"
    FOLLOWUP_ANSWER   = "No"
    try:
        msgs = [{"role": m["role"], "content": m["content"]} for m in messages]
        a_idxs = [i for i, m in enumerate(msgs) if m.get("role") == "assistant"]
        if not a_idxs:
            raise ValueError("no assistant turn to judge")
        msgs = msgs[: a_idxs[-1] + 1]                # trim to the last assistant turn being judged
        try:
            tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        except Exception:                            # fold a leading system turn (gemma)
            norm, pending = [], None
            for m in msgs:
                if m["role"] == "system" and not norm:
                    pending = m["content"]; continue
                if not norm and m["role"] == "user" and pending is not None:
                    norm.append({"role": "user", "content": pending + "\n\n" + m["content"]}); pending = None; continue
                norm.append(m)
            if pending is not None:
                norm = [{"role": "user", "content": pending}] + norm
            msgs = norm

        ext = msgs + [{"role": "user", "content": FOLLOWUP_QUESTION},
                      {"role": "assistant", "content": FOLLOWUP_ANSWER}]
        text = tokenizer.apply_chat_template(ext, tokenize=False, add_generation_prompt=False)
        content = FOLLOWUP_ANSWER.strip()
        enc = tokenizer(text, add_special_tokens=False)
        ids = enc["input_ids"]
        c = text.rfind(content)                      # char offset of the forced "No"
        if c == -1:
            raise ValueError("forced answer not found in rendered text")
        start = enc.char_to_token(c)
        start = 0 if start is None else start
        end = enc.char_to_token(c + len(content))
        end = len(ids) if end is None else end
        idx = end - 1                                # last token of "No"
        if idx < start:
            raise ValueError("empty answer span")
        if max_len and len(ids) > max_len:           # left-trim; the follow-up sits at the end
            cut = len(ids) - max_len
            ids, idx = ids[cut:], idx - cut
            if idx < 0:
                raise ValueError("answer trimmed away")
        idx = max(0, min(idx, len(ids) - 1))
        return ids, (idx, idx + 1)                   # 1-token span -> the forced "No" token
    except Exception as e:
        print(f"[warn] followup_preprocess fell back on a row: {type(e).__name__}: {e}")
        try:
            ids = tokenizer.apply_chat_template(
                [{"role": m["role"], "content": m["content"]} for m in messages],
                tokenize=True, add_generation_prompt=False, return_dict=True)["input_ids"]
        except Exception:
            ids = tokenizer(" ".join(str(m.get("content", "")) for m in messages),
                            add_special_tokens=False)["input_ids"] or [tokenizer.eos_token_id or 0]
        if max_len and len(ids) > max_len:
            ids = ids[len(ids) - max_len:]
        last = max(0, len(ids) - 1)
        return ids, (last, last + 1)


def detect(model, model_id, lora_id, batch, *, direction, bias, score_std, layer):
    """(B,) deception probabilities in [0,1], computed remotely at one layer. Identical to the
    mean probe's `detect` -- only the response span differs (the forced "No" token, a 1-token
    span, instead of the whole response), so `pool_response` returns exactly that token."""
    with model.trace({"input_ids": batch.input_ids,
                      "attention_mask": batch.attention_mask}):
        h = decoder_layers(model)[layer].output            # (B, seq, hidden) OR a tuple
        if isinstance(h, tuple):                            # some blocks (e.g. gemma-3) return (hidden, ...)
            h = h[0]
        feats = batch.pool_response(h)                      # (B, hidden)
        dev = feats.device
        s = (feats.float() @ direction.to(dev) - bias) / score_std
        # No tracer.stop(): the session is re-run remotely on NDIF, where stop()'s
        # EarlyStopException does not unwind cleanly and HANGS gemma's nested VLM forward.
        # We just let the forward finish and read the layer we already captured -- the proven
        # remote pattern (cf. the successful pooled_probe submission). Cost is negligible here.
        scores = torch.sigmoid(s).cpu().save()             # .cpu() so the cross-batch cat works if sharded
    return scores


## 3. Run one session and write `submission.csv`

In [ ]:
import time
import numpy as np
import torch
from util import build_model, decoder_layers, load_examples

# followup used to score the whole dataset in ONE remote model.session (run_full_session).
# That has no wall-clock guard: on a cold/slow gemma-BASE organism the single session runs
# past the sandbox timeout and SIGKILLs -> EVERY row lost (the rc=124 gemma failure). We keep
# the identical forward-pass read (layer LAYER output, pooled at the forced "No" token, NO
# tracer.stop() -- the proven remote pattern), but run it PER BATCH as a separate remote job
# with cold-start retry + a time budget, so a slow gemma run scores what it can and neutral-
# fills the rest instead of losing everything. Non-gemma organisms are unaffected in behaviour.
FOLLOWUP_TIME_BUDGET_S = float(os.environ.get("FOLLOWUP_TIME_BUDGET_S", 1500))
NB_START = time.time()

idx, scores = [], []
if HAVE_PROBE:
    BATCH_SIZE = 4 if "120B" in DS_MODEL else 8
    MODEL_OK = False
    try:
        model = build_model(DS_MODEL, util._norm_lora(DS_LORA))
        tokenizer = model.tokenizer
        pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
        layers_mod = decoder_layers(model)
        _DIRECTION = torch.tensor(DIRECTION)
        MODEL_OK = True
    except Exception as e:
        print(f"[warn] could not build model {DS_MODEL} ({type(e).__name__}: {e}); neutral for this dataset")

    if MODEL_OK:
        ds = load_examples(DATASET_NAME)
        n = len(ds) if LIMIT is None else min(LIMIT, len(ds))
        messages = ds["messages"][:n]
        all_index = list(ds["index"])[:n]

        # Tokenize LOCALLY with the probe's exact preprocessing (forced-"No" span).
        toks, spans = [], []
        for m in messages:
            ids_, span = followup_preprocess(list(m), tokenizer, MAX_SEQ_LEN)
            toks.append(ids_); spans.append(span)
        order = sorted(range(len(toks)), key=lambda i: len(toks[i]))   # short -> long padding
        n_batches = -(-len(order) // BATCH_SIZE)

        loop_start = time.time(); attempted = 0
        for bi, b0 in enumerate(range(0, len(order), BATCH_SIZE)):
            now = time.time(); avg = (now - loop_start) / attempted if attempted else 0.0
            if now - NB_START > FOLLOWUP_TIME_BUDGET_S or (attempted and now - NB_START + avg > FOLLOWUP_TIME_BUDGET_S):
                print(f"  time budget hit ({now - NB_START:.0f}s); {len(order) - b0} rows -> neutral")
                break
            attempted += 1
            bpos = order[b0:b0 + BATCH_SIZE]
            brows = [toks[i] for i in bpos]; bspans = [spans[i] for i in bpos]
            w = max(len(r) for r in brows)
            input_ids = torch.full((len(brows), w), pad_id, dtype=torch.long)
            attn = torch.zeros((len(brows), w), dtype=torch.long)
            for ri, r in enumerate(brows):
                input_ids[ri, :len(r)] = torch.tensor(r); attn[ri, :len(r)] = 1
            try:
                def _run():
                    with model.trace({"input_ids": input_ids, "attention_mask": attn}, remote=True):
                        h = layers_mod[LAYER].output                 # (B, seq, hidden) OR a tuple
                        if isinstance(h, tuple):                     # gemma-3 blocks return (hidden, ...)
                            h = h[0]
                        feats = torch.stack([h[k, s:e].mean(0) for k, (s, e) in enumerate(bspans)])
                        dev = feats.device
                        sc = torch.sigmoid((feats.float() @ _DIRECTION.to(dev) - BIAS) / SCORE_STD)
                        sc = sc.float().cpu().save()                 # .save() inside, read AFTER the block
                    return sc
                sc = util._remote_retry(_run, what=f"followup b{bi + 1}/{n_batches}")
                arr = np.asarray(sc.cpu().numpy(), dtype=float).reshape(-1)
                for j, i in enumerate(bpos):
                    idx.append(all_index[i]); scores.append(float(arr[j]))
                print(f"  batch {bi + 1}/{n_batches}")
            except Exception as e:
                print(f"  batch {bi + 1}/{n_batches} FAILED ({type(e).__name__}: {e}); "
                      f"skipping {len(bpos)} rows -> neutral")
                continue
        print(f"scored {len(scores)} rows across {attempted} batch(es)")
else:
    print("[warn] no probe -> neutral score for every row")

# One prediction per eval row, no duplicate/missing index (the real grader is strict -- unlike
# --dry's partial scoring). finalize_submission covers every eval index, collapses duplicates,
# and fills any unscored row (failed/over-budget batch, no probe) with a neutral 0.5.
util.finalize_submission(DATASET_NAME, idx, scores, limit=LIMIT)
